In [83]:
import pandas as pd
df = pd.read_excel("khdl_score_k39.xlsx")
df_monhoc = pd.read_excel("monhoc.xlsx")

# **Tiền xử lý dữ liêu**

In [84]:
# Ép kiểu cột MSSV sang kiểu chuỗi (string)
df["MSSV"] = df["MSSV"].astype(str)

In [85]:
so_mon_ban_dau = len(df.columns[2:])

In [86]:
# Cac mon so luong Null lon nhat, khong tinh mon Tu chon (1)
mon_tu_chon = df_monhoc[df_monhoc['Loại học phần'] == 'Tự Chọn']['Tên học phần'].tolist()
null_counts = df.isnull().sum()
mon_88_null = null_counts[null_counts == 88].index.tolist()
mon_xoa_88 = [m for m in mon_88_null if m not in mon_tu_chon]

In [87]:
# Danh sach cac mon GDTC va GDQP (2)
mon_gdtc_qp = [col for col in df.columns if 'GDTC' in col or 'Học phần' in col]
mon_gdtc_qp

['Học phần 1 (Đường lối quốc phòng và an ninh của Đảng Cộng sản Việt Nam)',
 'Học phần 2 (Công tác quốc phòng và an ninh của Đảng Cộng sản Việt nam)',
 'Học phần 3 (Quân sự chung)',
 'Học phần 4 (Kỹ thuật chiến đấu bộ binh và chiến thuật)',
 'Học phần GDTC 1',
 'Học phần GDTC 2',
 'Học phần GDTC 3',
 'Học phần GDTC 4',
 'Học phần GDTC 5']

In [88]:
# Danh sach cac mon chua hoc (3)
mon_nghiep_vu = [
    'Unnamed: 0',
    'An toàn bảo mật thông tin trong kinh doanh',
    'Chủ nghĩa xã hội khoa học',
    'Chuẩn công nghệ thông tin đầu vào',
    'Chuẩn ngoại ngữ đầu vào',
    'Lịch sử Đảng Cộng sản Việt Nam',
    'Thực tập cuối khóa chuyên ngành Khoa học trong kinh doanh',
    'Tiếng anh chuyên ngành Khoa học dữ liệu trong kinh doanh',
    'Khóa luận tốt nghiệp chuyên ngành Khoa học dữ liệu trong kinh doanh',
    'Trực quan hóa dữ liệu',
    'Phân tích dữ liệu cho tài chính'
]

In [89]:
# Xoa cac cot chua cac mon trong (1), (2), (3)
tat_ca_mon_xoa = list(set(mon_xoa_88 + mon_gdtc_qp + mon_nghiep_vu))
tat_ca_mon_xoa = [m for m in tat_ca_mon_xoa if m in df.columns]
df = df.drop(columns=tat_ca_mon_xoa)

In [90]:
print(f"Số môn đã xóa: {len(tat_ca_mon_xoa)}")

Số môn đã xóa: 20


In [91]:
df.columns[2:]

Index(['Chuỗi khối', 'Cơ sở dữ liệu', 'Đạo đức và văn hóa doanh nghiệp',
       'Giải thuật ứng dụng trong kinh doanh',
       'Hệ hoạch định nguồn lực doanh nghiệp', 'Hệ quản trị cơ sở dữ liệu',
       'Hệ thống thông tin quản lý', 'Học máy', 'Khai phá dữ liệu',
       'Kho dữ liệu và hệ hỗ trợ ra quyết định',
       'Kinh tế chính trị Mác - Lênin', 'Kinh tế học quốc tế',
       'Kinh tế học vi mô', 'Kinh tế học vĩ mô', 'Kinh tế lượng',
       'Lập trình hướng đối tượng', 'Lập trình Python cho phân tích dữ liệu',
       'Lập trình R trong thống kê', 'Logic ứng dụng trong kinh doanh',
       'Lý thuyết tài chính - tiền tệ', 'Lý thuyết xác suất và thống kê toán',
       'Nguyên lý kế toán', 'Nguyên lý Marketing',
       'Nhập ngành Khoa học dữ liệu trong kinh doanh', 'Phân tích dữ liệu lớn',
       'Phân tích dữ liệu mạng xã hội', 'Phân tích kinh doanh',
       'Phân tích tài chính doanh nghiệp', 'Pháp luật đại cương',
       'Phương pháp nghiên cứu khoa học', 'Phương pháp tối ưu trong 

In [92]:
# Loai bo cac sinh vien khong theo hoc
cols_mon = df.columns[2:]  
df['so_mon_null'] = df[cols_mon].isnull().sum(axis=1)
nguong = len(cols_mon) * 0.5
sv_bo_hoc = df[df['so_mon_null'] > nguong][['Họ Tên', 'so_mon_null']]
print(f"Sinh viên bị loại (bỏ học): {len(sv_bo_hoc)}")
df = df[df['so_mon_null'] <= nguong].drop(columns=['so_mon_null'])

Sinh viên bị loại (bỏ học): 4


In [93]:
# Xoa cac dong chua cac mon trong (1), (2), (3) cua df_monhoc
df_monhoc = df_monhoc[df_monhoc['Tên học phần'].isin(df.columns)]

In [94]:
# VT = 0 diem
df = df.replace("VT", 0)
# CT = Na
df = df.replace("CT", float("nan"))

# **Wide Format --> Long Format**

In [95]:
cols_info = ['MSSV', 'Họ Tên']
cols_mon = [col for col in df.columns if col not in cols_info]

df_long = df.melt(
    id_vars=cols_info,
    value_vars=cols_mon,
    var_name='Tên học phần',
    value_name='Điểm'
)

In [96]:
df_long

,MSSV,Họ Tên,Tên học phần,Điểm
0,30239230003,Nguyễn Thị Thùy An,Chuỗi khối,8.0
1,30239230005,Đào Việt Anh,Chuỗi khối,8.5
2,30239230009,Nguyễn Hồng Anh,Chuỗi khối,8.1
3,30239230020,Tôn Thất Gia Bảo,Chuỗi khối,8.1
4,30239230026,Nguyễn Ngọc Kim Cương,Chuỗi khối,8.0
...,...,...,...,...
3607,30239230292,Lâm Tuấn Vũ,Tư tưởng Hồ Chí Minh,9.3
3608,30239230293,Tào Quang Vũ,Tư tưởng Hồ Chí Minh,8.8
3609,30239230296,Trần Thị Ngọc Vy,Tư tưởng Hồ Chí Minh,7.7
3610,30239230297,Trần Thúy Vy,Tư tưởng Hồ Chí Minh,7.6


In [97]:
df_long["Điểm"] = df_long["Điểm"].astype(float)

In [100]:

df_long.to_excel("data_score.xlsx", index = False)
df_monhoc.to_excel("monhoc_daxuli.xlsx", index = False)